# MindAI on Google Colab — Cloud Inference

Run a MindAI brain on a free Colab T4 GPU and chat with it from the **Web UI** on your local computer (Cloud mode).

Two ways the brain can be used remotely:

1. **Cloud-mode chat** *(this notebook)* — both brain and world live on Colab. Your local Web UI sends prompts/files; tokens stream back. Endpoint: `wss://…/chat`.
2. **Remote-GPU training** *(legacy)* — brain on Colab, dataset/world on your PC. Endpoint: `wss://…/ws`. Run with `python main_agent.py --remote …`.

**Steps**
1. Runtime → Change runtime type → **GPU (T4)**.
2. Cell 1: clone + install.
3. Cell 2: optional ngrok auth-token.
4. Cell 3: start server. Copy the public URL.
5. On your local PC: open the Web UI, click **New chat → Cloud**, paste the URL.

In [ ]:
# --- Cell 1: setup ---------------------------------------------------------
import os, subprocess

REPO = 'https://github.com/mellson19/MindAI.git'   # ← change to your fork if needed
if not os.path.exists('MindAI'):
    subprocess.run(['git', 'clone', '--depth=1', REPO], check=True)
%cd MindAI

!pip install -q fastapi uvicorn websockets msgpack msgpack-numpy pyngrok scipy tiktoken edge-tts

# Optional — restore previously saved models from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !rm -rf models && cp -r /content/drive/MyDrive/mindai_models models

In [ ]:
# --- Cell 2: ngrok auth-token (optional but recommended) ------------------
# Free token: https://dashboard.ngrok.com/get-started/your-authtoken
# Without a token the tunnel closes after 2 hours.
import os
os.environ['NGROK_AUTHTOKEN'] = ''   # ← paste your token here

In [ ]:
# --- Cell 3: pre-create one or more models --------------------------------
# Models are stored in models/<id>/.  When you create a new chat in the Web UI
# (Cloud mode), the dropdown will show every model in this directory.
from pathlib import Path
import json, time

def make_model(model_id: str, display_name: str):
    d = Path('models') / model_id
    d.mkdir(parents=True, exist_ok=True)
    meta_p = d / 'meta.json'
    if not meta_p.exists():
        meta_p.write_text(json.dumps({
            'name':      display_name,
            'created':   int(time.time()),
            'last_used': 0,
        }, indent=2))
    print(f'  · {model_id}  ({display_name})')

make_model('default',     'Default')
make_model('alice',       'Alice — chat companion')
make_model('lab',         'Lab — fresh research brain')
print('models/ ready')

In [ ]:
# --- Cell 4: launch server with public ngrok tunnel -----------------------
# Blocks until interrupted.  In the output you'll see:
#     >>> URL для друга: wss://abc123.ngrok-free.app
# Paste that URL into the Web UI's Cloud-mode form.
!python server.py --ngrok

## Saving models back to Google Drive

When you stop the server, the active model auto-saves to `models/<id>/`.
Copy the whole `models/` directory back so you don't lose progress when the
Colab runtime is recycled:

```python
from google.colab import drive
drive.mount('/content/drive')
!cp -r models /content/drive/MyDrive/mindai_models
```

## Endpoints exposed by `server.py`

| Endpoint | Used by |
|---|---|
| `GET  /` | health check |
| `GET  /models` | Web UI Cloud-mode picker |
| `WS   /chat` | Web UI Cloud-mode chat (full pipeline server-side) |
| `WS   /ws` | legacy `--remote` mode (world stays on user's PC) |
| `POST /save` | manual save trigger |
| `GET  /weights/download` | download `models/` as zip |

## Important: each Cloud-mode chat is bound to ONE model

MindAI's neurons mutate during inference (every reply changes the synapses).
Switching the model mid-conversation would mean swapping in a brain with
completely different weights — so the Web UI doesn't allow it. To get an
independent learning trajectory, **duplicate** an existing model in the Models tab
before starting a new chat.